# Test

In [1]:
print("Tanis Testing-Chamber")

Tanis Testing-Chamber


# Mini-POC

## Daten abrufen/Erstellen

In [ ]:
import requests
import pandas as pd

# 1. Parameter für die Abfrage definieren
stichtag = "2024-07-01"
# Die URL setzt sich aus der Serveradresse und dem Stichtag-Parameter zusammen
url = f"http://127.0.0.1:8000/api/data?stichtag={stichtag}"

# 2. Server anfragen
print(f"Rufe Daten vom Server ab: {url}")
response = requests.get(url)

# 3. Antwort verarbeiten
if response.status_code == 200:
    # Das vom Server gelieferte JSON direkt wieder in einen DataFrame umwandeln
    data_json = response.json()
    
    if "error" in data_json:
         print("Server meldet einen Fehler:", data_json["error"])
    else:
        df_server = pd.DataFrame(data_json)
        print("\n--- Empfangene Daten ---")
        print(df_server.to_string())
else:
    print(f"Fehler bei der Verbindung. HTTP-Statuscode: {response.status_code}")

In [ ]:
import pandas as pd
import os

FILE_NAME = "historisierung_poc.csv"

# Prüfen, ob die Datei schon existiert. Wenn nicht, mit Dummy-Daten erstellen.
if not os.path.exists(FILE_NAME):
    initial_data = [
        {"kunden_id": 1001, "alter": 25, "tarif": "Basis", "status": "Aktiv", "valid_from": "2023-01-01", "valid_to": "2024-05-31"},
        {"kunden_id": 1001, "alter": 26, "tarif": "Premium", "status": "Aktiv", "valid_from": "2024-06-01", "valid_to": "2025-12-31"},
        {"kunden_id": 1001, "alter": 27, "tarif": "Premium", "status": "Gekündigt", "valid_from": "2026-01-01", "valid_to": "2099-12-31"},
        {"kunden_id": 1002, "alter": 40, "tarif": "Basis", "status": "Aktiv", "valid_from": "2022-03-15", "valid_to": "2025-01-10"},
        {"kunden_id": 1002, "alter": 43, "tarif": "Basis", "status": "Ruhend", "valid_from": "2025-01-11", "valid_to": "2099-12-31"}
    ]
    df_init = pd.DataFrame(initial_data)
    df_init.to_csv(FILE_NAME, index=False)
    print(f"Datei '{FILE_NAME}' wurde neu angelegt.")
else:
    print(f"Datei '{FILE_NAME}' gefunden und wird verwendet.")

# Daten in den Arbeitsspeicher laden und Datumstypen setzen
# df_hist = pd.read_csv(FILE_NAME)
# df_hist['valid_from'] = pd.to_datetime(df_hist['valid_from'])
# df_hist['valid_to'] = pd.to_datetime(df_hist['valid_to'])

## Neuen Datensatz einpflegen

In [ ]:
# def get_data_as_of(df: pd.DataFrame, as_of_date: str) -> pd.DataFrame:
#     """Filtert den historischen Datensatz für einen exakten Stichtag."""
#     target_date = pd.to_datetime(as_of_date)
#     mask = (df['valid_from'] <= target_date) & (df['valid_to'] >= target_date)
#     return df[mask].reset_index(drop=True)

def add_record(df: pd.DataFrame, new_record: dict, file_name: str) -> pd.DataFrame:
    """
    Fügt einen neuen Datensatz zum DataFrame hinzu und speichert in der CSV.
    Gibt den aktualisierten DataFrame zurück.
    """
    # Neuen Datensatz in DataFrame umwandeln
    df_new = pd.DataFrame([new_record])
    
    # Datumstypen für den neuen Datensatz sicherstellen
    df_new['valid_from'] = pd.to_datetime(df_new['valid_from'])
    df_new['valid_to'] = pd.to_datetime(df_new['valid_to'])
    
    # An den bestehenden DataFrame anhängen
    df_updated = pd.concat([df, df_new], ignore_index=True)
    
    # In CSV speichern (Datum als String speichern für sauberes Format)
    df_updated.to_csv(file_name, index=False)
    print(f"Datensatz für Kunden-ID {new_record['kunden_id']} hinzugefügt und in {file_name} gespeichert.")
    
    return df_updated

### Neuen Datensatz anlegen

In [ ]:
# 1. Einen neuen Datensatz definieren (z.B. ein komplett neuer Kunde)
neuer_kunde = {
    "kunden_id": 1006, 
    "alter": 33, 
    "tarif": "Student", 
    "status": "Aktiv", 
    "valid_from": "2026-05-15", 
    "valid_to": "2099-12-31" # 2099-12-31 steht wieder für "aktuell gültig"
}

# 2. Datensatz hinzufügen (df_hist wird dabei überschrieben, damit wir aktuell bleiben)
df_hist = add_record(df_hist, neuer_kunde, FILE_NAME)

### Zeitpunkt festlegen/Zeitreise

In [ ]:
# 3. Abfrage testen, um zu prüfen, ob der neue Kunde im Datensatz ist
stichtag = "2026-06-01"
df_ergebnis = get_data_as_of(df_hist, stichtag)

print(f"\n--- Datenstand zum Stichtag {stichtag} ---")
print(df_ergebnis.to_string())

## Datensatz überschreiben

In [ ]:
def update_customer_data(df: pd.DataFrame, kunden_id: int, new_attributes: dict, file_name: str) -> pd.DataFrame:
    """
    Simuliert die Integrationsschicht: Schließt den alten Datensatz 
    und öffnet einen neuen (SCD Typ 2).
    """
    now = pd.Timestamp.now().normalize() # Heutiges Datum ohne Uhrzeit
    
    # 1. Den aktuell gültigen Datensatz des Kunden finden
    mask_active = (df['kunden_id'] == kunden_id) & (df['valid_to'] == pd.to_datetime("2099-12-31"))
    
    if not df[mask_active].empty:
        # 2. Alten Datensatz abschließen (Gültigkeit endet heute)
        # Hinweis: In der Praxis oft 'gestern', um Überlappungen zu vermeiden.
        df.loc[mask_active, 'valid_to'] = now - pd.Timedelta(days=1)
        
        # Vorlage für neuen Datensatz aus den alten Daten erstellen (um unveränderte Werte zu behalten)
        old_record = df[mask_active].iloc[0].to_dict()
        new_record = old_record.copy()
    else:
        # Falls Kunde neu ist, leeres Template nutzen
        print(f"Kunde {kunden_id} nicht gefunden. Erstelle neuen Stammdatensatz.")
        new_record = {"kunden_id": kunden_id}

    # 3. Neue Attribute und Metadaten setzen
    new_record.update(new_attributes)
    new_record['valid_from'] = now
    new_record['valid_to'] = pd.to_datetime("2099-12-31")

    # 4. An DataFrame anhängen und speichern
    df_updated = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)
    df_updated.to_csv(file_name, index=False)
    
    print(f"Update für ID {kunden_id} durchgeführt.")
    return df_updated

### Änderung anlegen

In [ ]:
# --- Beispielaufruf ---
# Wir ändern für Kunde 1002 den Tarif auf 'Premium' und den Status auf 'Aktiv'
neue_werte = {"tarif": "Basis", "status": "Aktiv"}
df_hist = update_customer_data(df_hist, 1002, neue_werte, FILE_NAME)

# Kontrolle: Zeige alle Zeilen für diesen Kunden
print(df_hist[df_hist['kunden_id'] == 1002])